# 🌲 Random Forest — Ensemble Methods

> **Part of:** Basics → Ensemble_Methods → Bagging

---

## 📌 What is Bagging?

**Bagging (Bootstrap Aggregating)** is an ensemble technique where:
- Multiple models are trained **independently** on different random subsets of data
- Their predictions are **combined** (voting for classification, average for regression)
- It **reduces variance** and helps avoid overfitting

```
Original Dataset
      │
  ┌───┴───┐
Bootstrap Bootstrap Bootstrap
Sample 1  Sample 2  Sample 3
   │         │         │
 Tree 1    Tree 2    Tree 3
   └─────────┴─────────┘
         Majority Vote
         Final Prediction
```

---

## 🌲 What is Random Forest?

Random Forest is an **improved version of Bagging** that:
- Uses **Decision Trees** as base learners
- Adds **feature randomness** — each tree only considers a random subset of features at each split
- This makes trees more **diverse and uncorrelated**

| Feature | Bagging | Random Forest |
|---|---|---|
| Base Learner | Any model | Decision Tree |
| Feature Selection | All features | Random subset |
| Tree Correlation | High | Low |
| Performance | Good | Better |


---
## 📦 Step 1 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported successfully!')

---
## 📊 Step 2 — Load & Explore Dataset

We use the **Breast Cancer dataset** from sklearn:
- 569 samples, 30 features
- Binary classification: Malignant (0) vs Benign (1)

In [ ]:
# Load dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

print('Dataset Shape:', X.shape)
print('Classes:', data.target_names)
print('\nClass Distribution:')
print(y.value_counts())

X.head()

---
## ✂️ Step 3 — Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing  samples : {X_test.shape[0]}')

---
## 🧪 Step 4 — Bagging Classifier

In [ ]:
# Single Decision Tree (baseline)
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
dt_acc = accuracy_score(y_test, dt.predict(X_test))

# Bagging Classifier
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.8,
    random_state=42
)
bag.fit(X_train, y_train)
bag_acc = accuracy_score(y_test, bag.predict(X_test))

print(f'Single Decision Tree Accuracy : {dt_acc:.4f}')
print(f'Bagging Classifier Accuracy   : {bag_acc:.4f}')
print(f'\n📈 Improvement from Bagging   : +{(bag_acc - dt_acc):.4f}')

---
## 🌲 Step 5 — Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,      # number of trees
    max_depth=5,           # max depth of each tree
    max_features='sqrt',   # features per split = sqrt(total features)
    min_samples_split=2,
    random_state=42
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print('Random Forest Accuracy:', accuracy_score(y_test, y_pred))
print('\n📋 Classification Report:')
print(classification_report(y_test, y_pred, target_names=data.target_names))

---
## 📉 Step 6 — Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=data.target_names,
    cmap='Blues', ax=ax
)
plt.title('Random Forest — Confusion Matrix', fontsize=14)
plt.tight_layout()
plt.show()

---
## 📊 Step 7 — Feature Importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=data.feature_names)
top10 = importances.sort_values(ascending=False).head(10)

plt.figure(figsize=(9, 5))
sns.barplot(x=top10.values, y=top10.index, palette='viridis')
plt.title('Top 10 Feature Importances — Random Forest', fontsize=14)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Top 5 Features:')
print(top10.head())

---
## ⚙️ Step 8 — Effect of n_estimators

In [ ]:
n_trees = [1, 5, 10, 20, 50, 100, 200]
train_scores, test_scores = [], []

for n in n_trees:
    model = RandomForestClassifier(n_estimators=n, random_state=42)
    model.fit(X_train, y_train)
    train_scores.append(model.score(X_train, y_train))
    test_scores.append(model.score(X_test, y_test))

plt.figure(figsize=(9, 5))
plt.plot(n_trees, train_scores, 'o-', label='Train Accuracy', color='steelblue')
plt.plot(n_trees, test_scores,  's-', label='Test Accuracy',  color='coral')
plt.xlabel('Number of Trees (n_estimators)')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Number of Trees', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 🔁 Step 9 — Cross Validation

In [ ]:
rf_final = RandomForestClassifier(n_estimators=100, random_state=42)
cv_scores = cross_val_score(rf_final, X, y, cv=5, scoring='accuracy')

print('Cross-Validation Scores (5-Fold):')
for i, score in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {score:.4f}')
print(f'\nMean  : {cv_scores.mean():.4f}')
print(f'Std   : {cv_scores.std():.4f}')

---
## ✅ Step 10 — Model Comparison Summary

In [ ]:
rf_final.fit(X_train, y_train)
rf_acc = accuracy_score(y_test, rf_final.predict(X_test))

results = pd.DataFrame({
    'Model': ['Decision Tree', 'Bagging', 'Random Forest'],
    'Test Accuracy': [dt_acc, bag_acc, rf_acc]
})

plt.figure(figsize=(7, 4))
sns.barplot(data=results, x='Model', y='Test Accuracy', palette=['#ff6b6b','#ffd93d','#6bcb77'])
plt.ylim(0.9, 1.0)
plt.title('Model Comparison', fontsize=14)
plt.ylabel('Accuracy')
for i, row in results.iterrows():
    plt.text(i, row['Test Accuracy'] + 0.001, f"{row['Test Accuracy']:.4f}", ha='center', fontsize=11)
plt.tight_layout()
plt.show()

print(results.to_string(index=False))

---
## 📝 Key Hyperparameters

| Parameter | What it controls | Tip |
|---|---|---|
| `n_estimators` | Number of trees | More = better (but slower) |
| `max_depth` | Max depth per tree | Controls overfitting |
| `max_features` | Features per split | `'sqrt'` for classification |
| `min_samples_split` | Min samples to split node | Higher = more regularization |
| `bootstrap` | Use bootstrap samples | Default True |

---

## 🚀 When to Use Random Forest

✅ Large datasets with many features  
✅ When you need feature importance  
✅ When interpretability is not critical  
✅ As a strong baseline before trying XGBoost  

❌ When you need full interpretability  
❌ Very high-dimensional sparse data (use linear models)  

---
**Next:** `03_Boosting_XGBoost.ipynb` →